# Supplemental RT Inference

Runs batch inference for a supplemental Regular Transformer (RT) checkpoint against a matching RRUFF benchmark HDF5 and reports top-1/3/5 accuracy. No W&B login required for execution; the notebook does include an optional W&B-API helper for fetching architecture from the original training run.

Backs **Table S5 / S6 (RT real-RRUFF ablation)** in the supplement (`tab:real_ablation`). Source script: `flash_attn_version/test_inference.py`.

## Environment

```bash
conda activate paper-ai-diffraction-train-eval
pip install -e .
```

Required packages (already in `environment-train-eval.yml`): `torch`, `h5py`, `numpy`, `tqdm`. The `wandb` package is optional and only needed for the W&B-API architecture lookup cell.

## Checkpoints

Download supplemental RT checkpoints from Zenodo and place them under `external/checkpoints/`. RT checkpoint mapping is still being finalised — see `reproducibility/checkpoint_manifest.csv` (RT rows pending) and the `CHECKPOINT_CONFIGS` dict below.

In [ ]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

MODELS_DIR = REPO_ROOT / 'external' / 'checkpoints'

# ---------------------------------------------------------------------------
# Per-checkpoint configs for the supplemental RT runs.
#
# Architecture defaults shared across all RT checkpoints in this notebook
# (verified from flash_attn_version/cross_test.py and inference_config.json):
#   embed_dim=256  ff_dim=128  mlp_units=256  pos_encoding='rope'  num_classes=99
#
# Per-checkpoint variation: depth, num_heads, spec_length, matching RRUFF HDF5.
# Fill in W&B run IDs and matching benchmark HDFs for each Table S5/S6 row;
# placeholders here mirror the user's local training rig.
# ---------------------------------------------------------------------------

CHECKPOINT_CONFIGS = {

    # ── Table S5 (RT): real-RRUFF performance, depth=8 / depth=6 family ────
    # spec_length=8501; 5–90°; eval on RRUFF low-bkg full (cleaned) intensity
    'yv1m76u6': dict(  # RT-Balanced       (training-distribution checkpoint)
        ckpt='xrd_model_yv1m76u6.pth',
        spec_length=8501, depth=8, num_heads=4,
        rruff_h5='/scratch/10471/ashetty4428/data/RRUFF_low_bkg_full_intensity_cleaned.hdf5',
    ),
    '4hv17ttu': dict(  # RT-ICSD
        ckpt='xrd_model_4hv17ttu.pth',
        spec_length=8501, depth=6, num_heads=4,
        rruff_h5='/scratch/10471/ashetty4428/data/RRUFF_low_bkg_full_intensity_cleaned.hdf5',
    ),
    'hwixtnv7': dict(  # RT-RRUFF          (also used for Fig S? attention overlay)
        ckpt='xrd_model_hwixtnv7.pth',
        spec_length=8501, depth=6, num_heads=4,
        rruff_h5='/scratch/10471/ashetty4428/data/RRUFF_low_bkg_full_intensity_cleaned.hdf5',
    ),
    'mq1l94p7': dict(  # RT-Augmented
        ckpt='xrd_model_mq1l94p7.pth',
        spec_length=8501, depth=6, num_heads=4,
        rruff_h5='/scratch/10471/ashetty4428/data/RRUFF_low_bkg_full_intensity_cleaned.hdf5',
    ),

    # ── Additional ablation row (Table S6 RT): 16-heads variant ────────────
    # See flash_attn_version/inference_config.json (model_note='1-epoch-noise-16-heads')
    '7brb1pir': dict(
        ckpt='xrd_model_7brb1pir.pth',
        spec_length=8501, depth=6, num_heads=16,
        rruff_h5='/scratch/10471/ashetty4428/data/RRUFF_low_bkg_full_intensity_cleaned.hdf5',
    ),
}

RUN_ID = 'hwixtnv7'
cfg         = CHECKPOINT_CONFIGS[RUN_ID]
ckpt_file   = cfg['ckpt']
RRUFF_H5    = cfg['rruff_h5']
SPEC_LENGTH = cfg['spec_length']
DEPTH       = cfg['depth']
NUM_HEADS   = cfg['num_heads']
CHECKPOINT  = str(MODELS_DIR / ckpt_file)

print(f'Run ID     : {RUN_ID}')
print(f'Checkpoint : {CHECKPOINT}')
print(f'Benchmark  : {RRUFF_H5}')
print(f'spec_length: {SPEC_LENGTH}  depth: {DEPTH}  num_heads: {NUM_HEADS}')

### Optional: pull architecture from W&B

If you want to verify the architecture against the original training run rather than the hardcoded `CHECKPOINT_CONFIGS` entry, run the cell below. It requires `wandb login` and read access to the `nist-berkeley-ai-diffraction/ai-diffraction` project. Skip it for offline reproduction.

In [ ]:
USE_WANDB_LOOKUP = False
WANDB_PROJECT = 'nist-berkeley-ai-diffraction/ai-diffraction'

if USE_WANDB_LOOKUP:
    import wandb
    api = wandb.Api()
    run = api.run(f'{WANDB_PROJECT}/{RUN_ID}')
    cfg_wb = run.config
    def _val(key):
        v = cfg_wb.get(key)
        return v if not isinstance(v, dict) else (v.get('value') or v.get('values', [None])[0])
    DEPTH       = _val('depth') or DEPTH
    NUM_HEADS   = _val('num_heads') or NUM_HEADS
    SPEC_LENGTH = _val('spec_length') or SPEC_LENGTH
    print(f'W&B-resolved: depth={DEPTH}  num_heads={NUM_HEADS}  spec_length={SPEC_LENGTH}')

In [ ]:
MODEL_CONFIG = dict(
    spec_length  = SPEC_LENGTH,
    num_output   = 99,
    embed_dim    = 256,
    ff_dim       = 128,
    depth        = DEPTH,
    num_heads    = NUM_HEADS,
    mlp_units    = 256,
    dropout      = 0.0,
    pos_encoding = 'rope',
    # Flash-Attention is fine for inference; falls back to eager if unavailable.
    use_flash_attn = True,
)

NUM_CLASSES = 99
BATCH_SIZE  = 128
NUM_WORKERS = 4
START_COL   = 1
END_COL     = SPEC_LENGTH

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
from collections import OrderedDict
from tqdm.auto import tqdm

from paper_ai_diffraction.core.rt_model import transformer_model
from paper_ai_diffraction.core.dataset import get_dataloaders_test

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Match training precision (RT was trained at fp16/bf16).
dtype = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32

model = transformer_model(**MODEL_CONFIG)

ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt.get('model', ckpt)
clean = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
model.load_state_dict(clean, strict=False)
model = model.to(device=device, dtype=dtype).eval()
print(f'Loaded: {ckpt_file}  (dtype={dtype})')

In [ ]:
_, _, test_loader, _, _ = get_dataloaders_test(
    h5_file       = RRUFF_H5,
    batch_size    = BATCH_SIZE,
    num_classes   = NUM_CLASSES,
    num_workers   = NUM_WORKERS,
    prefetch_factor = 2,
    start_col     = START_COL,
    end_col       = END_COL,
    label_mode    = 'categorical',
)
print(f'Test samples: {len(test_loader.dataset)}')

In [ ]:
top1 = top3 = top5 = total = 0

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc='Evaluating'):
        inputs  = inputs.to(device=device, dtype=dtype)
        targets = targets.to(device)
        outputs = model(inputs)
        total  += targets.size(0)

        _, topk = outputs.topk(5, dim=1)
        t = targets.view(-1, 1)
        top1 += topk[:, :1].eq(t).any(1).sum().item()
        top3 += topk[:, :3].eq(t).any(1).sum().item()
        top5 += topk[:, :5].eq(t).any(1).sum().item()

print(f'\nResults for {RUN_ID} on {Path(RRUFF_H5).name}')
print(f'  Total samples : {total}')
print(f'  Top-1         : {100 * top1 / total:.2f}%')
print(f'  Top-3         : {100 * top3 / total:.2f}%')
print(f'  Top-5         : {100 * top5 / total:.2f}%')